# Repository Distribution Analysis
This notebook analyzes the distribution and characteristics of repositories in the dataset.

In [1]:
import pandas as pd
import ast


df_repo = pd.read_csv("../data/repo_characteristics.csv")
df_repo["doc_files"] = df_repo["doc_files"].map(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else []
)
df_repo["created_year"] = pd.to_datetime(df_repo["created_at"]).dt.year

## Process Commit Author Information
This section iterates through commit path files to extract unique author information for each commit, handling both GitHub logins and author names.

In [2]:
import json
import tqdm
from pathlib import Path
import pandas as pd

commits_path_dir = Path("../data/commits_path")

records = []
for filepath in tqdm.tqdm(commits_path_dir.iterdir()):
    sha = filepath.name  # markdown sha, no extension
    try:
        with open(filepath, encoding="utf-8") as f:
            commits = json.load(f)
        # collect unique author logins for this markdown file
        authors = set()
        for commit in commits:
            # prefer GitHub login (stable), fallback to author name
            login = (commit.get("author") or {}).get("login")
            if login:
                authors.add(login)
            else:
                name = (commit.get("commit") or {}).get("author", {}).get("name")
                if name:
                    authors.add(name)
        records.append({"sha": sha, "authors": list(authors)})
    except Exception as e:
        print(f"Error reading {filepath}: {e}")

df_commits_path = pd.DataFrame(records)

13360it [00:09, 1467.45it/s]


## Filter Data for Analysis
This section filters the repository and commit data to include only those entries present in the classification dataset, ensuring consistency across analyses.

In [3]:
df_classifications = pd.read_csv(
    "../data/classifications/classifications_for_analysis.csv"
)
df_repo_filtered = df_repo[
    df_repo["full_name"].isin(df_classifications["repository_full_name"])
]
df_commits_filtered = df_commits_path[
    df_commits_path["sha"].isin(df_classifications["sha"])
]

This cell identifies repositories present in the classification data but missing from the repository metadata.

In [4]:
df_classifications[
    ~df_classifications["repository_full_name"].isin(df_repo["full_name"])
]["repository_full_name"].unique()

<StringArray>
[                           'Chailllee/2025Courses_LLM',
                                           'Kcato1/two',
                      'Renan04lima/mapa-do-combustivel',
             'jonathan-nascimento51/glpi_dashboard_cau',
                                'lalalune/ainex-remote',
 'poisontr33s/psychonoir-kontrapunkt-large-file-holder',
               'sethdford/aws-sam-java-personalization']
Length: 7, dtype: str

In [5]:
# Calculates the number of unique authors in the filtered commit data
df_commits_filtered["authors"].explode().nunique()

899

In [6]:
for files in df_repo["doc_files"].to_list():
    filtered_files = [f for f in files if not f.startswith(".specstory/")]
    # print(filtered_files)